In [ ]:
import os
import numpy as np
from fractions import Fraction
from math import floor, gcd, log
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFTGate
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

token = os.getenv('IBM_QUANTUM_TOKEN')
instance = os.getenv('IBM_QUANTUM_INSTANCE')

try:
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        instance=instance,
        overwrite=True)
except Exception as e:
    print(f"Error: {e}")

[Quantum Fourier transform and phase estimation](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/qft#qpe-with-more-precision-more-)

[Shor's algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/shors-algorithm)

## Why $N=15$ and $a=2$? (The "Cheat" Explained)

In educational implementations of Shor's algorithm, choosing $N=15$ and $a=2$ is a strategic "shortcut" to make the experiment viable on current quantum hardware.

### 1. Hardware Constraints
Factoring larger numbers requires $2n + 3$ qubits (where $n$ is the bit-length of $N$). By using $N=15$, we keep the circuit within the 12-13 qubit range, which fits modern QPUs and avoids excessive noise (decoherence) that would destroy the calculation.



### 2. Simplified Modular Multiplication
With $a=2$ and $N=15$, the period $r$ is very short ($r=4$).
* $2^1 \pmod{15} = 2$
* $2^2 \pmod{15} = 4$
* $2^3 \pmod{15} = 8$
* $2^4 \pmod{15} = 1$ (Period found!)

This allows us to implement modular multiplication using simple **SWAP gates** instead of deep, complex modular exponentiation circuits that current hardware cannot yet handle accurately.



### 3. Guaranteed Success

In a real-world scenario, the base $a$ is chosen at random. If the resulting period $r$ is odd or if it is not true that at least one prime factor of $N$ divides $\text{gcd}(a^{r/2} \pm 1, N)$, the algorithm must be restarted with a different $a$. By choosing **$a=2$**, we ensure that the conditions for success are met on the first value of $a$ we have selected, thus guaranteeing a successful factorization of $N=15$ without needing to repeat the experiment with different values of $a$.

In [ ]:
def M2mod15():
    """
    M2 (mod 15)
    """
    b = 2
    U = QuantumCircuit(4)
 
    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)
 
    U = U.to_gate()
    U.name = f"M_{b}"
 
    return U

In [ ]:
# Get the M2 operator
M2 = M2mod15()
 
# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M2, inplace=True)
print("M2 (mod 15) operator:")
circ.decompose(reps=2).draw(output="mpl", fold=-1)

### Mathematical Derivation of the $M_4$ Operator

The goal of $M_4$ is to implement the modular multiplication $|y\rangle \to |4y \pmod{15}\rangle$. Since $4 = 2^2$, this operator performs a **double cyclic shift** in the binary register.

#### 1. Action of $M_4$ on Cycle States
We first define how $M_4$ transforms the four states of our cycle in decimal representation:

* $M_4 |1\rangle = |4\rangle$
* $M_4 |2\rangle = |8\rangle$
* $M_4 |4\rangle = |16 \pmod{15}\rangle = |1\rangle$
* $M_4 |8\rangle = |32 \pmod{15}\rangle = |2\rangle$

#### 2. Binary Mapping (Double Shift)
Let's rewrite these actions in binary representation ($|q_3 q_2 q_1 q_0\rangle$):

* $M_4 |0001\rangle = |0100\rangle$
* $M_4 |0010\rangle = |1000\rangle$
* $M_4 |0100\rangle = |0001\rangle$
* $M_4 |1000\rangle = |0010\rangle$

#### 3. Deconstruction into SWAP Gates
To achieve this "double jump" (bit 0 moves to 2, bit 1 moves to 3), we can use two independent SWAP gates. Unlike $M_2$, where bits move to adjacent positions, $M_4$ swaps bits across two positions:

$$M_4 = \text{SWAP}(0,2) \text{SWAP}(1,3)$$

In [ ]:
def M4mod15():
    """
    M4 (mod 15)
    """
    b = 4
    U = QuantumCircuit(4)
 
    U.swap(1, 3)
    U.swap(0, 2)
 
    U = U.to_gate()
    U.name = f"M_{b}"
 
    return U

In [ ]:
# Get the M4 operator
M4 = M4mod15()
 
# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M4, inplace=True)
print("M4 (mod 15) operator:")
circ.decompose(reps=2).draw(output="mpl", fold=-1)

In [ ]:
def a2kmodN(a, k, N):
    """Compute a^{2^k} (mod N) by repeated squaring"""
    for _ in range(k):
        a = int(np.mod(a**2, N))
    return a

In [ ]:
# Order finding problem for N = 15 with a = 2
N = 15
a = 2
 
# Number of qubits
num_target = floor(log(N - 1, 2)) + 1  # for modular exponentiation operators
num_control = 2 * num_target  # for enough precision of estimation
 
# List of M_b operators in order
k_list = range(num_control)
b_list = [a2kmodN(2, k, 15) for k in k_list]
 
# Initialize the circuit
control = QuantumRegister(num_control, name="C")
target = QuantumRegister(num_target, name="T")
output = ClassicalRegister(num_control, name="out")
circuit = QuantumCircuit(control, target, output)
 
# Initialize the target register to the state |1>
circuit.x(num_control)
 
# Add the Hadamard gates and controlled versions of the
# multiplication gates
for k, qubit in enumerate(control):
    circuit.h(k)
    b = b_list[k]
    if b == 2:
        circuit.compose(
            M2mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    elif b == 4:
        circuit.compose(
            M4mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    else:
        continue  # M1 is the identity operator
 
# Apply the inverse QFT to the control register
circuit.compose(QFTGate(num_control).inverse(), qubits=control, inplace=True)

# Measure the control register
circuit.measure(control, output)
 
print(f"Circuit for order finding with N={N} and a={a}:")
circuit.draw("mpl", fold=-1)

If you want to run the circuit on the Aer simulator:

In [ ]:
shots = 1024

backend = AerSimulator()
job = backend.run(circuit.decompose(), shots=shots)
result = job.result()

result = job.result()
counts = result.get_counts(circuit)

If you want to run the circuit on a real quantum computer:

In [ ]:
shots = 1024

service = QiskitRuntimeService()
print("Selecting backend...")
backend = service.least_busy(simulator=False, operational=True)
print(f"Running on backend: {backend.name}")

pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
 
transpiled_circuit = pm.run(circuit)
 
# Sampler primitive to obtain the probability distribution
sampler = Sampler(backend)
 
# Turn on dynamical decoupling with sequence XpXm
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"
# Enable gate twirling
sampler.options.twirling.enable_gates = True

job = sampler.run([transpiled_circuit], shots=shots)

job_id = job.job_id()
print(f"Job sumbitted. ID: {job_id}") # Wait for the job to complete and then retrieve the results

Retrieve the results from the job run on the real quantum computer:

In [ ]:
retrieved_job = service.job(job_id)
current_status = retrieved_job.status()
if current_status == 'DONE':
    result = retrieved_job.result()
    counts = result[0].data["out"].get_counts()
    print("Job completed successfully!")
else:
    print(f"Job is not completed yet. Current status: {current_status}")

In [ ]:
display(plot_histogram(counts, figsize=(35, 5)))

# Dictionary of bitstrings and their counts to keep
counts_keep = {}
# Threshold to filter
threshold = np.max(list(counts.values())) / 2
 
for key, value in counts.items():
    if value > threshold:
        counts_keep[key] = value
 
print(counts_keep)

In [ ]:
FACTOR_FOUND = False

for attempt, bitstring in enumerate(counts_keep):
    print(f"ATTEMPT {attempt + 1}:")
    # Here, we get the bitstring by iterating over outcomes
    # of a previous hardware run with multiple shots.
    # Instead, we can also perform a single-shot measurement
    # here in the loop.
    
    # Trova la fase dalla misura
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control)  # phase = k / r
    print(f"Phase: theta = {phase}")

    # Guess the order from phase
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator  # order = r
    print(f"Order of {a} modulo {N} estimated as: r = {r}")

    if phase != 0:
        # Guesses for factors are gcd(a^{r / 2} ± 1, N)
        if r % 2 == 0:
            x = pow(a, r // 2, N)
            
            for guess in [gcd(x - 1, N), gcd(x + 1, N)]:
                if 1 < guess < N:
                    FACTOR_FOUND = True
                    print(f"*** Non-trivial factor found: {guess} ***")
                    break
    
    if FACTOR_FOUND:
        break

if not FACTOR_FOUND:
	print("No factor found, try with another value of a.")

## Quantum vs Classical Factorization

### 1. Complexity Speedup: Polynomial vs Sub-exponential
The core advantage of Shor's algorithm lies in its efficiency compared to the best-known classical methods.

* **Classical (Sieves Method):** The fastest classical algorithm today is the **General Number Field Sieve (GNFS)**. Its complexity is **sub-exponential**: $\text{O}\left( e^{1.9 K^{1/3} (\log K)^{2/3}} \right)$, where $K$ is the number of digits. As $N$ grows, the time required explodes, making RSA encryption secure for now.
* **Quantum (Shor's Algorithm):** Shor's is a probabilistic algorithm that operates in **polynomial** steps: $Poly(K)$. More precisely, it can be optimized to $O(K^2 \log K \log \log K)$, which is significantly more efficient than $O(K^3)$. This transition from sub-exponential to polynomial is an **exponential speedup**.



### 2. Mechanism: Period Finding via QFT
Shor's algorithm mathematically reduces the problem of factorization to finding the **period ($r$)** of a modular function $f(x) = a^x \pmod N$.

* **The Quantum Step:** While a classical computer would need to check values one by one (a very slow process), a quantum computer uses **Quantum Phase Estimation (QPE)** and the **Inverse Quantum Fourier Transform (IQFT)**. These tools use constructive interference to highlight the period $r$ among all possible states simultaneously.
* **Final Factorization:** Once the period $r$ is extracted from the quantum measurement, the factors of $N$ are easily calculated on a classical computer using the Greatest Common Divisor (**GCD**) formula: $\text{gcd}(a^{r/2} \pm 1, N)$.